## FROM CORPUS XLSX TO JSON

In [1]:
#import necessary libraries
import json
import random
import pandas as pd
from collections import defaultdict

**Emotion JSONL**

In [ ]:
#This script reads messages and emotions from an Excel file and writes them as structured prompt-response pairs 
#in a JSONL format for emotion classification.
df = pd.read_excel("corpus.xlsx") #data input file name=corpus.xlsx
examples=0 #used to know number of rows 
# Iterate over each row of the DataFrame
with open("corpus_EMOCION.jsonl", "w", encoding="utf-8") as f: #data output file name=corpus_EMOCION.jsonl
    for _, row in df.iterrows():
        # Add each row as a separate message
        data = {
            "messages": [
                {
                    "role": "system", #cuestion ask to the model
                    "content": "What is the emotion of the following text? Respond with 'Anger', 'Disgust', 'Fear', 'Happy', 'Sad', 'Neutral', 'Surprised' or 'Empathy'." 
                },
                {
                    "role": "user",
                    "content": row["TEXT"] #column title where the message is located
                },
                {
                    "role": "assistant",
                    "content": row["Emotions"] #column title where the emotion is located
                }
            ]
        }
        f.write(json.dumps(data)+"\n")
        examples+=1
# to resume
print("Num examples:", int(examples))
print("File saved as corpus_EMOCION.jsonl")

**Polarity JSONL**

In [ ]:
# This script converts messages and their sentiment polarity from an Excel file 
#into a JSONL format for polarity classification prompts.
df = pd.read_excel("corpus.xlsx") #data input file name=corpus.xlsx
examples=0 #used to know number of rows 
# Iterate over each row of the DataFrame
with open("corpus_POLARIDAD.jsonl", "w", encoding="utf-8") as f: #data output file name=corpus_POLARIDAD.jsonl
    for _, row in df.iterrows():
        #  Add each row as a separate message
        data = {
            "messages": [
                {
                    "role": "system",#cuestion ask to the model
                    "content": "What is the polarity of the following text? Respond with 'Positive', 'Negative' or Neutral'." 
                },
                {
                    "role": "user",
                    "content": row["TEXT"] #column title where the message is located
                },
                {
                    "role": "assistant",
                    "content": row["Polarity"]  #column title where the polarity is located
                }
            ]
        }
        f.write(json.dumps(data)+"\n")
        examples+=1
# to resume
print("Num examples:", int(examples))
print("File saved as corpus_POLARIDAD.jsonl")

## Data verification

In [19]:
#This line sets the file path for the emotion-labeled dataset stored in JSONL format.
data_path = "corpus_EMOCION.jsonl"

In [9]:
#This line sets the file path for the polarity-labeled dataset stored in JSONL format.
data_path = "corpus_POLARIDAD.jsonl"

In [ ]:
# This code reads a JSONL dataset file into a list of JSON objects and prints the total number of examples.
with open(data_path, 'r', encoding='utf-8') as f:
    dataset = [json.loads(line) for line in f if line.strip()]

# Initial dataset stats
print("Num examples:", len(dataset))

In [ ]:
# This script validates the structure of each JSONL example by checking required fields and roles, and reports formatting errors if found.
format_errors = defaultdict(int)

for ex in dataset:
    if not isinstance(ex, dict):
        format_errors["data_type"] += 1
        continue
        
    messages = ex.get("messages", None)
    if not messages:
        format_errors["missing_messages_list"] += 1
        continue
        
    for message in messages:
        if "role" not in message or "content" not in message:
            format_errors["message_missing_key"] += 1
        
        if any(k not in ("role", "content", "name", "function_call", "weight") for k in message):
            format_errors["message_unrecognized_key"] += 1
        
        if message.get("role", None) not in ("system", "user", "assistant", "function"):
            format_errors["unrecognized_role"] += 1
            
        content = message.get("content", None)
        function_call = message.get("function_call", None)
        
        if (not content and not function_call) or not isinstance(content, str):
            format_errors["missing_content"] += 1
    
    if not any(message.get("role", None) == "assistant" for message in messages):
        format_errors["example_missing_assistant_message"] += 1

if format_errors:
    print("Found errors:")
    for k, v in format_errors.items():
        print(f"{k}: {v}")
else:
    print("No errors found")

## Splitting data into training, verification and testing

In [4]:
#This function splits a JSONL dataset into training (70%), validation (15%), and test (15%) sets, 
#shuffles the data, and writes each split to separate files.
#Imputs: (data file, training_file, validation_file, test_file)
def split_json(input_file, output_file_Training, output_file_validation, output_file_test):
    with open(input_file, 'r') as f: 
        data= [json.loads(line) for line in f]
        total_messages =len(data)

   # Calculate the split sizes
    n_train = int(total_messages * 0.70)
    n_val = int(total_messages * 0.15)
    n_test = total_messages - n_train - n_val  # ensures total is preserved

    #makes a copy of the data, sorts it randomly
    shuffled_data=data[:]
    random.shuffle(shuffled_data)
    
    # Split the data
    data_train = shuffled_data[:n_train]
    data_val = shuffled_data[n_train:n_train + n_val]
    data_test = shuffled_data[n_train + n_val:]

    # Write to output files
    with open(output_file_Training, 'w') as f_train:
        for item in data_train:
            json.dump(item, f_train)
            f_train.write('\n') #add a new line after each json object

    with open(output_file_validation, 'w') as f_val:
        for item in data_val:
            json.dump(item, f_val)
            f_val.write('\n') #add a new line after each json object

    with open(output_file_test, 'w') as f_test:
        for item in data_test:
            json.dump(item, f_test)
            f_test.write('\n') #add a new line after each json object
            

In [12]:
#This line calls the `split_json` function to divide the emotion-labeled dataset into training, validation, and test files.
split_json("corpus_EMOCION.jsonl", "training_Emocion.jsonl", "validation_Emocion.jsonl","test_Emocion.jsonl")

In [5]:
#This line calls the `split_json` function to divide the polarity-labeled dataset into training, validation, and test files.
split_json("corpus_POLARIDAD.jsonl", "training_Polaridad.jsonl", "validation_Polaridad.jsonl","test_Polaridad.jsonl")

In [ ]:
#This code computes and prints polarity label counts for training, validation, and test JSONL files.
filenames = [
    "training_Polaridad.jsonl",
    "validation_Polaridad.jsonl",
    "test_Polaridad.jsonl"
]
for filename in filenames:
    counts = {
        "Positive": 0,
        "Negative": 0,
        "Neutral": 0
    }
    total = 0
    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            data = json.loads(line)
            messages = data.get("messages", [])
            assistant_msgs = [m for m in messages if m.get("role") == "assistant"]
            if assistant_msgs:
                polarity = assistant_msgs[-1].get("content")
                if polarity in counts:
                    counts[polarity] += 1
                    total += 1

    print(f"Polarity statistics for {filename}:")
    print(f"Total examples: {total}")
    for polarity, count in counts.items():
        print(f"{polarity}: {count}")
    print("-" * 40)

In [ ]:
#This script counts and displays the distribution of emotion labels in training, validation, and test JSONL files.
filenames = [
    "training_Emocion.jsonl", "validation_Emocion.jsonl","test_Emocion.jsonl"
]
for filename in filenames:
    counts = defaultdict(int)
    total = 0

    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            data = json.loads(line)
            messages = data.get("messages", [])
            assistant_msgs = [m for m in messages if m.get("role") == "assistant"]
            if assistant_msgs:
                emotion = assistant_msgs[-1].get("content")
                if emotion:
                    counts[emotion] += 1
                    total += 1

    print(f"Emotion statistics for {filename}:")
    print(f"Total examples: {total}")
    for emotion, count in counts.items():
        print(f"{emotion}: {count}")
    print("-" * 40)

## Convert the polarity test file from JSON format to Excel

In [32]:
#This code reads a polarity-labeled JSONL test file, extracts texts and labels, and saves them into an Excel file.
with open("test_Polaridad.jsonl", "r") as file: #imput file:test_Polaridad
    data = [json.loads(line) for line in file]

texts = [] #empty list
polarity = [] #empty list

for dato in data: #read only the messages and polarity in the jsonl file
    texts.append(dato["messages"][1]["content"])
    polarity.append(dato["messages"][2]["content"])


df = pd.DataFrame({"MESSAGE": texts, "POLARITY": polarity}) #save de data in column form

df.to_excel("test_Polaridad.xlsx", index=False) #save into excel

## Convert the emotions test file from JSON format to Excel

In [33]:
#This code reads an emotion test JSONL file, extracts messages and emotions, then exports the data to an Excel spreadsheet.
with open("test_Emocion.jsonl", "r") as file:
    data = [json.loads(line) for line in file]

texts = [] #empty list
emotion = [] #empty list

for dato in data: #read only the messages and emotion in the jsonl file
    texts.append(dato["messages"][1]["content"])
    emotion.append(dato["messages"][2]["content"])


df = pd.DataFrame({"MESSAGE": texts, "EMOTION": emotion}) #save de data in column form

df.to_excel("test_Emocion.xlsx", index=False) #save into excel